# 📊 Online Retail II — Customer Lifetime Value & Cohort Retention Analysis

## 1. Import Libraries & Load Dataset

In [47]:
import pandas as pd
import numpy as np

In [48]:
df = pd.read_csv("online_retail_II.csv")

## 2. Initial Data Exploration
Checking shape, info, and first few rows of the dataset to understand its structure.

In [49]:
print(df.shape)

(1067371, 8)


In [50]:
print(df.info)

<bound method DataFrame.info of         Invoice StockCode                          Description  Quantity  \
0        489434     85048  15CM CHRISTMAS GLASS BALL 20 LIGHTS        12   
1        489434    79323P                   PINK CHERRY LIGHTS        12   
2        489434    79323W                  WHITE CHERRY LIGHTS        12   
3        489434     22041         RECORD FRAME 7" SINGLE SIZE         48   
4        489434     21232       STRAWBERRY CERAMIC TRINKET BOX        24   
...         ...       ...                                  ...       ...   
1067366  581587     22899         CHILDREN'S APRON DOLLY GIRL          6   
1067367  581587     23254        CHILDRENS CUTLERY DOLLY GIRL          4   
1067368  581587     23255      CHILDRENS CUTLERY CIRCUS PARADE         4   
1067369  581587     22138        BAKING SET 9 PIECE RETROSPOT          3   
1067370  581587      POST                              POSTAGE         1   

                 InvoiceDate  Price  Customer ID       

In [51]:
print(df.head(5))

  Invoice StockCode                          Description  Quantity  \
0  489434     85048  15CM CHRISTMAS GLASS BALL 20 LIGHTS        12   
1  489434    79323P                   PINK CHERRY LIGHTS        12   
2  489434    79323W                  WHITE CHERRY LIGHTS        12   
3  489434     22041         RECORD FRAME 7" SINGLE SIZE         48   
4  489434     21232       STRAWBERRY CERAMIC TRINKET BOX        24   

           InvoiceDate  Price  Customer ID         Country  
0  2009-12-01 07:45:00   6.95      13085.0  United Kingdom  
1  2009-12-01 07:45:00   6.75      13085.0  United Kingdom  
2  2009-12-01 07:45:00   6.75      13085.0  United Kingdom  
3  2009-12-01 07:45:00   2.10      13085.0  United Kingdom  
4  2009-12-01 07:45:00   1.25      13085.0  United Kingdom  


## 3. Data Cleaning

### 3.1 Flag Cancelled Transactions
Invoices starting with 'C' represent cancelled orders.

In [52]:
df['Invoice'] = df['Invoice'].astype(str)

In [53]:
df['Is_Cancelled'] = df['Invoice'].str.startswith('C')

In [54]:
print(df['Is_Cancelled'].value_counts())

Is_Cancelled
False    1047877
True       19494
Name: count, dtype: int64


### 3.2 Handle Missing Customer IDs
Rows without a Customer ID cannot be used for customer-level analysis (RFM/CLV), so they are removed.

In [55]:
print(df['Customer ID'].isnull().sum())

243007


In [56]:
df = df.dropna(subset = ['Customer ID'])

In [57]:
print(df['Customer ID'].isnull().sum())

0


### 3.3 Split Sales vs Cancelled Transactions
Separating positive quantity (actual sales) from negative quantity (returns/cancellations) for independent analysis.

In [58]:
df_sales = df[df['Quantity'] > 0].copy()

In [59]:
df_cancelled = df[df['Quantity'] < 0].copy()

In [60]:
print("Total Sales Rows:", df_sales.shape[0])
print("Total Cancelled Rows:", df_cancelled.shape[0])

Total Sales Rows: 805620
Total Cancelled Rows: 18744


### 3.4 Remove Invalid Price Records
Removing rows with zero or negative unit price, as these represent data entry errors.

In [61]:
print(df_sales[df_sales['Price'] <= 0].shape[0])

71


In [62]:
df_sales = df_sales[df_sales['Price'] > 0]

In [63]:
print("After cleaning, total rows:", df_sales.shape[0])

After cleaning, total rows: 805549


### 3.5 Remove Duplicate Records

In [64]:
print("Duplicate rows:", df_sales.duplicated().sum())

Duplicate rows: 26124


In [65]:
df_sales = df_sales.drop_duplicates()

In [66]:
print("After removing duplicates:", df_sales.shape[0])

After removing duplicates: 779425


### 3.6 Feature Engineering — Total Price per Transaction
Calculating revenue per line item (Quantity × Price) for further analysis.

In [67]:
df_sales['TotalPrice'] = df_sales['Quantity'] * df_sales['Price']

In [68]:
df_sales['InvoiceDate'] = pd.to_datetime(df_sales['InvoiceDate'])

df_sales.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Is_Cancelled,TotalPrice
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,False,83.4
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,False,81.0
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,False,81.0
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,False,100.8
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,False,30.0


## 4. Final Data Validation
Confirming the cleaned dataset has no missing values and consistent data types before proceeding to analysis.

In [69]:
print("Final shape:", df_sales.shape)

Final shape: (779425, 10)


In [70]:
print(df_sales.isnull().sum()) 

Invoice         0
StockCode       0
Description     0
Quantity        0
InvoiceDate     0
Price           0
Customer ID     0
Country         0
Is_Cancelled    0
TotalPrice      0
dtype: int64


In [71]:
print(df_sales.describe())

            Quantity                    InvoiceDate          Price  \
count  779425.000000                         779425  779425.000000   
mean       13.489370  2011-01-03 01:44:42.593475584       3.218488   
min         1.000000            2009-12-01 07:45:00       0.001000   
25%         2.000000            2010-07-02 14:39:00       1.250000   
50%         6.000000            2010-12-02 14:09:00       1.950000   
75%        12.000000            2011-08-01 13:44:00       3.750000   
max     80995.000000            2011-12-09 12:50:00   10953.500000   
std       145.855814                            NaN      29.676140   

         Customer ID     TotalPrice  
count  779425.000000  779425.000000  
mean    15320.360461      22.291823  
min     12346.000000       0.001000  
25%     13971.000000       4.950000  
50%     15247.000000      12.480000  
75%     16794.000000      19.800000  
max     18287.000000  168469.600000  
std      1695.692775     227.427075  


## 5. RFM Analysis (Recency, Frequency, Monetary)

Calculating RFM metrics for each customer to understand their purchasing behavior:
- **Recency:** How recently a customer made a purchase
- **Frequency:** How often they purchase
- **Monetary:** How much they spend in total

In [72]:
import datetime as dt

In [73]:
reference_date = df_sales['InvoiceDate'].max() + dt.timedelta(days=1)
print("Reference Date:", reference_date)

# RFM calculate 
rfm = df_sales.groupby('Customer ID').agg({
    'InvoiceDate': lambda x: (reference_date - x.max()).days,   # Recency
    'Invoice': 'nunique',                                        # Frequency
    'TotalPrice': 'sum'                                          # Monetary
}).reset_index()

# Columns rename
rfm.columns = ['CustomerID', 'Recency', 'Frequency', 'Monetary']

print(rfm.shape)
rfm.head()

Reference Date: 2011-12-10 12:50:00
(5878, 4)


,CustomerID,Recency,Frequency,Monetary
0,12346.0,326,12,77556.46
1,12347.0,2,8,4921.53
2,12348.0,75,5,2019.40
3,12349.0,19,4,4428.69
4,12350.0,310,1,334.40


In [74]:
rfm.describe()

,CustomerID,Recency,Frequency,Monetary
count,5878.000000,5878.000000,5878.000000,5878.000000
mean,15315.313542,201.331916,6.289384,2955.904095
std,1715.572666,209.338707,13.009406,14440.852688
min,12346.000000,1.000000,1.000000,2.950000
25%,13833.250000,26.000000,1.000000,342.280000
50%,15314.500000,96.000000,3.000000,867.740000
75%,16797.750000,380.000000,7.000000,2248.305000
max,18287.000000,739.000000,398.000000,580987.040000


## 6. RFM Scoring

Dividing customers into 4 quartiles for each metric (Recency, Frequency, Monetary) and assigning a score from 1-4. Combining these scores creates an RFM segment code for each customer.

In [75]:
# Recency
rfm['R_Score'] = pd.qcut(rfm['Recency'], 4, labels=[4, 3, 2, 1])

# Frequency: rank(method='first')
rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'), 4, labels=[1, 2, 3, 4])

# Monetary: 
rfm['M_Score'] = pd.qcut(rfm['Monetary'], 4, labels=[1, 2, 3, 4])


rfm['RFM_Score'] = rfm['R_Score'].astype(str) + rfm['F_Score'].astype(str) + rfm['M_Score'].astype(str)

rfm.head(10)

,CustomerID,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Score
0,12346.0,326,12,77556.46,2,4,4,244
1,12347.0,2,8,4921.53,4,4,4,444
2,12348.0,75,5,2019.40,3,3,3,333
3,12349.0,19,4,4428.69,4,3,4,434
4,12350.0,310,1,334.40,2,1,1,211
5,12351.0,375,1,300.93,2,1,1,211
6,12352.0,36,10,2849.84,3,4,4,344
7,12353.0,204,2,406.76,2,2,2,222
8,12354.0,232,1,1079.40,2,1,3,213
9,12355.0,214,2,947.61,2,2,3,223


In [76]:
rfm['RFM_Score'].value_counts().head(10)

RFM_Score
444    659
111    571
344    345
211    253
233    228
333    224
222    218
433    213
122    209
112    161
Name: count, dtype: int64

## 7. Customer Segmentation

Converting RFM scores into meaningful business segments to easily identify customer types — such as loyal customers, customers at risk of churning, and lost customers.

In [77]:
def segment_customer(row):
    if row['RFM_Score'] in ['444', '434', '443', '344']:
        return 'Champions'
    elif row['R_Score'] == '4' and row['F_Score'] in ['3', '4']:
        return 'Loyal Customers'
    elif row['R_Score'] in ['3', '4'] and row['F_Score'] in ['1', '2']:
        return 'Potential Loyalist'
    elif row['R_Score'] == '1' and row['F_Score'] in ['3', '4']:
        return 'At Risk'
    elif row['R_Score'] == '1' and row['F_Score'] == '1':
        return 'Lost Customers'
    else:
        return 'Others'

# Important: R_Score aur F_Score categorical hain, unhe string me convert karo comparison ke liye
rfm['R_Score'] = rfm['R_Score'].astype(str)
rfm['F_Score'] = rfm['F_Score'].astype(str)
rfm['M_Score'] = rfm['M_Score'].astype(str)

rfm['Segment'] = rfm.apply(segment_customer, axis=1)

print(rfm['Segment'].value_counts())
rfm.head(10)

Segment
Others                2467
Champions             1184
Potential Loyalist     894
Lost Customers         772
Loyal Customers        335
At Risk                226
Name: count, dtype: int64


,CustomerID,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Score,Segment
0,12346.0,326,12,77556.46,2,4,4,244,Others
1,12347.0,2,8,4921.53,4,4,4,444,Champions
2,12348.0,75,5,2019.40,3,3,3,333,Others
3,12349.0,19,4,4428.69,4,3,4,434,Champions
4,12350.0,310,1,334.40,2,1,1,211,Others
5,12351.0,375,1,300.93,2,1,1,211,Others
6,12352.0,36,10,2849.84,3,4,4,344,Champions
7,12353.0,204,2,406.76,2,2,2,222,Others
8,12354.0,232,1,1079.40,2,1,3,213,Others
9,12355.0,214,2,947.61,2,2,3,223,Others


## 8. Customer Lifetime Value (CLV) Calculation

Calculating two types of CLV:
- **Historical CLV:** Total revenue generated by the customer so far
- **Predicted CLV:** Estimated future value based on average order value and purchase frequency

In [78]:
# Historical CLV = Total Monetary Value 
rfm['Historical_CLV'] = rfm['Monetary']

# Average Order Value 
rfm['AOV'] = rfm['Monetary'] / rfm['Frequency']

# Predicted CLV 
# Formula: CLV = Average Order Value * Purchase Frequency * Estimated Lifespan
estimated_lifespan_years = 1
rfm['Predicted_CLV'] = rfm['AOV'] * rfm['Frequency'] * estimated_lifespan_years

rfm[['CustomerID', 'Segment', 'Historical_CLV', 'AOV', 'Predicted_CLV']].head(10)

,CustomerID,Segment,Historical_CLV,AOV,Predicted_CLV
0,12346.0,Others,77556.46,6463.038333,77556.46
1,12347.0,Champions,4921.53,615.191250,4921.53
2,12348.0,Others,2019.40,403.880000,2019.40
3,12349.0,Champions,4428.69,1107.172500,4428.69
4,12350.0,Others,334.40,334.400000,334.40
5,12351.0,Others,300.93,300.930000,300.93
6,12352.0,Champions,2849.84,284.984000,2849.84
7,12353.0,Others,406.76,203.380000,406.76
8,12354.0,Others,1079.40,1079.400000,1079.40
9,12355.0,Others,947.61,473.805000,947.61


In [79]:
rfm.sort_values('Predicted_CLV', ascending=False).head(10)

,CustomerID,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Score,Segment,Historical_CLV,AOV,Predicted_CLV
5692,18102.0,1,145,580987.04,4,4,4,444,Champions,580987.04,4006.807172,580987.04
2277,14646.0,2,151,528602.52,4,4,4,444,Champions,528602.52,3500.678940,528602.52
1789,14156.0,10,156,313437.62,4,4,4,444,Champions,313437.62,2009.215513,313437.62
2538,14911.0,1,398,291420.81,4,4,4,444,Champions,291420.81,732.213090,291420.81
5050,17450.0,8,51,244784.25,4,4,4,444,Champions,244784.25,4799.691176,244784.25
1331,13694.0,4,143,195640.69,4,4,4,444,Champions,195640.69,1368.116713,195640.69
5109,17511.0,3,60,172132.87,4,4,4,444,Champions,172132.87,2868.881167,172132.87
4061,16446.0,1,2,168472.50,4,2,4,424,Potential Loyalist,168472.50,84236.250000,168472.50
4295,16684.0,4,55,147142.77,4,4,4,444,Champions,147142.77,2675.323091,147142.77
68,12415.0,24,28,144458.37,4,4,4,444,Champions,144458.37,5159.227500,144458.37


## 9. Cohort Analysis

Grouping customers by the month of their first purchase (cohort) and tracking how many of them continued purchasing in subsequent months. This helps measure customer retention over time.

In [80]:
# transaction wise InvoiceMonth 
df_sales['InvoiceMonth'] = df_sales['InvoiceDate'].dt.to_period('M')

# customer purchase wise month (CohortMonth)
df_sales['CohortMonth'] = df_sales.groupby('Customer ID')['InvoiceDate'].transform('min').dt.to_period('M')

# CohortIndex = first purchase wise months transaction
df_sales['CohortIndex'] = (df_sales['InvoiceMonth'] - df_sales['CohortMonth']).apply(lambda x: x.n)

df_sales[['Customer ID', 'InvoiceDate', 'InvoiceMonth', 'CohortMonth', 'CohortIndex']].head(10)

,Customer ID,InvoiceDate,InvoiceMonth,CohortMonth,CohortIndex
0,13085.0,2009-12-01 07:45:00,2009-12,2009-12,0
1,13085.0,2009-12-01 07:45:00,2009-12,2009-12,0
2,13085.0,2009-12-01 07:45:00,2009-12,2009-12,0
3,13085.0,2009-12-01 07:45:00,2009-12,2009-12,0
4,13085.0,2009-12-01 07:45:00,2009-12,2009-12,0
5,13085.0,2009-12-01 07:45:00,2009-12,2009-12,0
6,13085.0,2009-12-01 07:45:00,2009-12,2009-12,0
7,13085.0,2009-12-01 07:45:00,2009-12,2009-12,0
8,13085.0,2009-12-01 07:46:00,2009-12,2009-12,0
9,13085.0,2009-12-01 07:46:00,2009-12,2009-12,0


In [81]:
# CohortMonth + CohortIndex 
cohort_data = df_sales.groupby(['CohortMonth', 'CohortIndex'])['Customer ID'].nunique().reset_index()

# Pivot table banao - rows me CohortMonth, columns me CohortIndex
cohort_pivot = cohort_data.pivot(index='CohortMonth', columns='CohortIndex', values='Customer ID')

cohort_pivot

CohortIndex,0,1,2,3,4,5,6,7,8,9,...,15,16,17,18,19,20,21,22,23,24
CohortMonth,,,,,,,,,,,,,,,,,,,,,
2009-12,955.0,337.0,319.0,406.0,363.0,343.0,360.0,327.0,321.0,346.0,...,289.0,251.0,289.0,270.0,248.0,244.0,301.0,291.0,389.0,188.0
2010-01,383.0,79.0,119.0,117.0,101.0,115.0,99.0,88.0,107.0,122.0,...,58.0,90.0,76.0,71.0,75.0,93.0,74.0,94.0,22.0,NaN
2010-02,374.0,89.0,84.0,109.0,92.0,75.0,72.0,107.0,95.0,103.0,...,75.0,60.0,61.0,54.0,86.0,86.0,61.0,22.0,NaN,NaN
2010-03,443.0,84.0,102.0,107.0,103.0,90.0,109.0,134.0,122.0,48.0,...,75.0,77.0,69.0,78.0,89.0,94.0,35.0,NaN,NaN,NaN
2010-04,294.0,57.0,57.0,48.0,54.0,66.0,81.0,77.0,31.0,32.0,...,46.0,41.0,44.0,53.0,66.0,17.0,NaN,NaN,NaN,NaN
2010-05,254.0,40.0,43.0,44.0,45.0,65.0,54.0,32.0,15.0,21.0,...,32.0,35.0,42.0,39.0,12.0,NaN,NaN,NaN,NaN,NaN
2010-06,270.0,47.0,51.0,55.0,62.0,77.0,34.0,24.0,22.0,32.0,...,33.0,36.0,55.0,14.0,NaN,NaN,NaN,NaN,NaN,NaN
2010-07,186.0,29.0,34.0,55.0,54.0,26.0,21.0,27.0,27.0,21.0,...,32.0,44.0,15.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2010-08,162.0,33.0,48.0,52.0,28.0,19.0,16.0,20.0,22.0,21.0,...,32.0,11.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 10. Retention Rate Calculation

Converting the cohort customer counts into percentages to visualize retention trends — showing what percentage of each cohort's customers remained active in subsequent months.

In [82]:
cohort_size = cohort_pivot.iloc[:, 0]


retention = cohort_pivot.divide(cohort_size, axis=0) * 100

retention.round(1)

CohortIndex,0,1,2,3,4,5,6,7,8,9,...,15,16,17,18,19,20,21,22,23,24
CohortMonth,,,,,,,,,,,,,,,,,,,,,
2009-12,100.0,35.3,33.4,42.5,38.0,35.9,37.7,34.2,33.6,36.2,...,30.3,26.3,30.3,28.3,26.0,25.5,31.5,30.5,40.7,19.7
2010-01,100.0,20.6,31.1,30.5,26.4,30.0,25.8,23.0,27.9,31.9,...,15.1,23.5,19.8,18.5,19.6,24.3,19.3,24.5,5.7,NaN
2010-02,100.0,23.8,22.5,29.1,24.6,20.1,19.3,28.6,25.4,27.5,...,20.1,16.0,16.3,14.4,23.0,23.0,16.3,5.9,NaN,NaN
2010-03,100.0,19.0,23.0,24.2,23.3,20.3,24.6,30.2,27.5,10.8,...,16.9,17.4,15.6,17.6,20.1,21.2,7.9,NaN,NaN,NaN
2010-04,100.0,19.4,19.4,16.3,18.4,22.4,27.6,26.2,10.5,10.9,...,15.6,13.9,15.0,18.0,22.4,5.8,NaN,NaN,NaN,NaN
2010-05,100.0,15.7,16.9,17.3,17.7,25.6,21.3,12.6,5.9,8.3,...,12.6,13.8,16.5,15.4,4.7,NaN,NaN,NaN,NaN,NaN
2010-06,100.0,17.4,18.9,20.4,23.0,28.5,12.6,8.9,8.1,11.9,...,12.2,13.3,20.4,5.2,NaN,NaN,NaN,NaN,NaN,NaN
2010-07,100.0,15.6,18.3,29.6,29.0,14.0,11.3,14.5,14.5,11.3,...,17.2,23.7,8.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2010-08,100.0,20.4,29.6,32.1,17.3,11.7,9.9,12.3,13.6,13.0,...,19.8,6.8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 11. Exporting Cleaned Data for SQL, Excel & Power BI

Saving the cleaned transactional data, RFM/CLV table, and cohort retention table as CSV files — these will be used to build the SQL database, Excel summary report, and Power BI dashboard.

In [83]:
# 1. Cleaned sales data
df_sales.to_csv("cleaned_online_retail.csv", index=False)

# 2. RFM + CLV table - customer-level analysis 
rfm.to_csv("rfm_clv_table.csv", index=False)

# 3. Cohort retention table - Power BI heatmap 
retention.to_csv("cohort_retention.csv")

print("✅ All files saved successfully!")
print("1. cleaned_online_retail.csv - Shape:", df_sales.shape)
print("2. rfm_clv_table.csv - Shape:", rfm.shape)
print("3. cohort_retention.csv - Shape:", retention.shape)

✅ All files saved successfully!
1. cleaned_online_retail.csv - Shape: (779425, 13)
2. rfm_clv_table.csv - Shape: (5878, 12)
3. cohort_retention.csv - Shape: (25, 25)


In [84]:
df_sales.columns.tolist()

['Invoice',
 'StockCode',
 'Description',
 'Quantity',
 'InvoiceDate',
 'Price',
 'Customer ID',
 'Country',
 'Is_Cancelled',
 'TotalPrice',
 'InvoiceMonth',
 'CohortMonth',
 'CohortIndex']

In [85]:
df_sales.to_csv("cleaned_online_retail.csv", index=False)
print("✅ File updated and saved!")

✅ File updated and saved!


In [86]:
# Retention table ke columns ko clearly rename karo
retention.columns = [f"Month_{i}" for i in retention.columns]

# Index (CohortMonth) ko bhi proper column bana do
retention_export = retention.reset_index()
retention_export = retention_export.rename(columns={'CohortMonth': 'CohortMonth'})

# Check karo naye column names
print(retention_export.columns.tolist())

# Dobara CSV save karo
retention_export.to_csv("cohort_retention.csv", index=False)
print("✅ File updated with proper column names!")

['CohortMonth', 'Month_0', 'Month_1', 'Month_2', 'Month_3', 'Month_4', 'Month_5', 'Month_6', 'Month_7', 'Month_8', 'Month_9', 'Month_10', 'Month_11', 'Month_12', 'Month_13', 'Month_14', 'Month_15', 'Month_16', 'Month_17', 'Month_18', 'Month_19', 'Month_20', 'Month_21', 'Month_22', 'Month_23', 'Month_24']
✅ File updated with proper column names!


In [87]:
# Customer ka purchase span nikalo (pehli aur aakhri purchase ke beech kitne din)
customer_span = df_sales.groupby('Customer ID')['InvoiceDate'].agg(['min', 'max'])
customer_span['Span_Days'] = (customer_span['max'] - customer_span['min']).dt.days
customer_span['Span_Days'] = customer_span['Span_Days'].replace(0, 1)  # 0 din wale ko 1 kar do (divide by zero avoid)

rfm = rfm.merge(customer_span[['Span_Days']], left_on='CustomerID', right_index=True)

# Purchase Frequency PER YEAR nikalo (ab ye total Frequency jaisa nahi hoga)
rfm['Frequency_Per_Year'] = rfm['Frequency'] / (rfm['Span_Days'] / 365)

# Ab Predicted CLV — agle 1 saal ka realistic estimate
estimated_future_years = 1
rfm['Predicted_CLV'] = rfm['AOV'] * rfm['Frequency_Per_Year'] * estimated_future_years

print(rfm[['CustomerID', 'Historical_CLV', 'Frequency_Per_Year', 'Predicted_CLV']].head(10))

   CustomerID  Historical_CLV  Frequency_Per_Year  Predicted_CLV
0     12346.0        77556.46           10.950000   70770.269750
1     12347.0         4921.53            7.263682    4468.553358
2     12348.0         2019.40            5.041436    2036.135359
3     12349.0         4428.69            2.561404    2835.915526
4     12350.0          334.40          365.000000  122056.000000
5     12351.0          300.93          365.000000  109839.450000
6     12352.0         2849.84           10.252809    2921.886517
7     12353.0          406.76            3.578431     727.781373
8     12354.0         1079.40          365.000000  393981.000000
9     12355.0          947.61            2.067989     979.823371


In [89]:
# Frequency_Per_Year ko realistic range me cap karo (max 52 baar/saal = weekly buyer se zyada nahi maanenge)
rfm['Frequency_Per_Year'] = rfm['Frequency_Per_Year'].clip(upper=52)

# Predicted CLV dobara calculate karo is capped value se
rfm['Predicted_CLV'] = rfm['AOV'] * rfm['Frequency_Per_Year'] * estimated_future_years

print(rfm[['CustomerID', 'Historical_CLV', 'Frequency_Per_Year', 'Predicted_CLV']].head(10))

   CustomerID  Historical_CLV  Frequency_Per_Year  Predicted_CLV
0     12346.0        77556.46           10.950000   70770.269750
1     12347.0         4921.53            7.263682    4468.553358
2     12348.0         2019.40            5.041436    2036.135359
3     12349.0         4428.69            2.561404    2835.915526
4     12350.0          334.40           52.000000   17388.800000
5     12351.0          300.93           52.000000   15648.360000
6     12352.0         2849.84           10.252809    2921.886517
7     12353.0          406.76            3.578431     727.781373
8     12354.0         1079.40           52.000000   56128.800000
9     12355.0          947.61            2.067989     979.823371


In [90]:
print(rfm[['Historical_CLV', 'Predicted_CLV']].describe())

       Historical_CLV  Predicted_CLV
count     5878.000000    5878.000000
mean      2955.904095    8379.641848
std      14440.852688   22831.909120
min          2.950000      19.958922
25%        342.280000    1310.274341
50%        867.740000    3156.197682
75%       2248.305000    8815.336667
max     580987.040000  691886.000000


In [91]:
rfm.to_csv("rfm_clv_table.csv", index=False)
print("✅ Updated with realistic Predicted CLV!")

✅ Updated with realistic Predicted CLV!
